In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

ROOT = Path(r"F:\THESIS\mtech_thesis_crop_yield_estimation")
PROCESSED = ROOT / "data" / "processed"

# load and merge
fields = gpd.read_file(PROCESSED / "yield_analysis_ready.gpkg")
sar = pd.read_csv(PROCESSED / "sar_monthly_features.csv").drop(columns=["year"], errors="ignore")
df = fields.merge(sar, on="uid", how="left")

sar_cols = [c for c in sar.columns if c.startswith(("VV_", "VH_", "ratio_"))]

# modeling set: no flag, has yield, complete SAR
d = df[(df["qc_flag"] == "") & (df["crop_yield"].notna()) & (df[sar_cols].notna().all(axis=1))].copy()
print("modeling fields (with SAR):", len(d))
print("SAR features:", len(sar_cols))

# spatial groups (same 200m clustering as baseline)
utm = d.to_crs(32643)
coords = np.c_[utm.geometry.centroid.x.values, utm.geometry.centroid.y.values]
tree = cKDTree(coords)
pairs = tree.query_pairs(r=200)
n = len(d)
r, c = [], []
for a, b in pairs:
    r += [a, b]; c += [b, a]
graph = csr_matrix((np.ones(len(r)), (r, c)), shape=(n, n))
_, geo_group = connected_components(graph, directed=False)
d = d.reset_index(drop=True)
d["geo_group"] = geo_group
print("spatial groups:", d["geo_group"].nunique())

modeling fields (with SAR): 1197
SAR features: 18
spatial groups: 804


In [3]:
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import GroupKFold
# from sklearn.metrics import r2_score, mean_squared_error
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LinearRegression
# from sklearn.ensemble import RandomForestRegressor

# # feature sets
# attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
# attr_num = ["area_acres", "year"]
# attr_features = attr_cat + attr_num

# y = d["crop_yield"].values
# groups = d["geo_group"].values
# gkf = GroupKFold(n_splits=5)

# # fill category blanks
# data = d.copy()
# for c in attr_cat:
#     data[c] = data[c].astype(str).fillna("Unknown")

# def run_sklearn(model, feature_cols, cat_cols, label):
#     X = data[feature_cols]
#     pre = ColumnTransformer([
#         ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in cat_cols if c in feature_cols]),
#         ("num", "passthrough", [c for c in feature_cols if c not in cat_cols]),
#     ])
#     pipe = Pipeline([("prep", pre), ("model", model)])
#     va_r2s, va_rms = [], []
#     for tr, va in gkf.split(X, y, groups):
#         pipe.fit(X.iloc[tr], y[tr])
#         p = pipe.predict(X.iloc[va])
#         va_r2s.append(r2_score(y[va], p))
#         va_rms.append(np.sqrt(mean_squared_error(y[va], p)))
#     print(f"{label:<38} VALID R2 {np.mean(va_r2s):6.3f} | RMSE {np.mean(va_rms):5.2f}")
#     return np.mean(va_r2s)

# def run_catboost(feature_cols, cat_cols, label):
#     X = data[feature_cols].copy()
#     cats = [c for c in cat_cols if c in feature_cols]
#     cat_idx = [X.columns.get_loc(c) for c in cats]
#     va_r2s, va_rms = [], []
#     for tr, va in gkf.split(X, y, groups):
#         m = CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_state=42, verbose=0)
#         m.fit(Pool(X.iloc[tr], y[tr], cat_features=cat_idx))
#         p = m.predict(X.iloc[va])
#         va_r2s.append(r2_score(y[va], p))
#         va_rms.append(np.sqrt(mean_squared_error(y[va], p)))
#     print(f"{label:<38} VALID R2 {np.mean(va_r2s):6.3f} | RMSE {np.mean(va_rms):5.2f}")
#     return np.mean(va_r2s)

# featuresets = [
#     ("attributes", attr_features, attr_cat),
#     ("SAR only",   sar_cols,       []),
#     ("attr + SAR", attr_features + sar_cols, attr_cat),
# ]

# print("Grouped 5-fold CV, n=1197, same fields\n" + "="*60)
# for fs_name, cols, cats in featuresets:
#     print(f"\n--- {fs_name} ---")
#     run_sklearn(LinearRegression(), cols, cats, "Linear Regression")
#     run_sklearn(RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), cols, cats, "Random Forest")
#     run_catboost(cols, cats, "CatBoost")

In [4]:
months = ["Jun", "Jul", "Aug", "Sep", "Oct", "Nov"]

# ---- DpRVIc per month (Bhogapurapu et al. 2022) ----
# q = VH_linear / VV_linear ; DpRVIc = q(q+3)/(q+1)^2
for m in months:
    vv_lin = 10 ** (d[f"VV_{m}"] / 10)
    vh_lin = 10 ** (d[f"VH_{m}"] / 10)
    q = vh_lin / vv_lin
    d[f"DpRVIc_{m}"] = (q * (q + 3)) / ((q + 1) ** 2)

# ---- delta_sigma per month: VV above the field's own dry baseline ----
# dry reference = the field's minimum VV across the season (its bare/driest month, in dB)
vv_cols = [f"VV_{m}" for m in months]
d["vv_dry_ref"] = d[vv_cols].min(axis=1)  # per-field driest month as baseline
for m in months:
    d[f"delta_sigma_{m}"] = d[f"VV_{m}"] - d["vv_dry_ref"]

dprvic_cols = [f"DpRVIc_{m}" for m in months]
delta_cols  = [f"delta_sigma_{m}" for m in months]
physics_cols = dprvic_cols + delta_cols

print("added physics features:", len(physics_cols))
print("DpRVIc range:", round(d[dprvic_cols].min().min(),3), "to", round(d[dprvic_cols].max().max(),3), "(should be 0 to 1)")
print("delta_sigma range:", round(d[delta_cols].min().min(),2), "to", round(d[delta_cols].max().max(),2), "dB")
print()
print("DpRVIc mean by month (should rise then fall, like a crop):")
print(d[dprvic_cols].mean().round(3).to_string())

added physics features: 12
DpRVIc range: 0.068 to 0.741 (should be 0 to 1)
delta_sigma range: 0.0 to 8.55 dB

DpRVIc mean by month (should rise then fall, like a crop):
DpRVIc_Jun    0.292
DpRVIc_Jul    0.353
DpRVIc_Aug    0.423
DpRVIc_Sep    0.476
DpRVIc_Oct    0.437
DpRVIc_Nov    0.374


In [5]:
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import GroupKFold
# from sklearn.metrics import r2_score, mean_squared_error
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import LinearRegression
# from sklearn.ensemble import RandomForestRegressor

# y = d["crop_yield"].values
# groups = d["geo_group"].values
# gkf = GroupKFold(n_splits=5)

# attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
# attr_num = ["area_acres", "year"]
# attr_features = attr_cat + attr_num

# data = d.copy()
# for c in attr_cat:
#     data[c] = data[c].astype(str).fillna("Unknown")

# def run_sklearn(model, cols, cat_cols):
#     X = data[cols]
#     pre = ColumnTransformer([
#         ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in cat_cols if c in cols]),
#         ("num", "passthrough", [c for c in cols if c not in cat_cols]),
#     ])
#     pipe = Pipeline([("prep", pre), ("model", model)])
#     r2s, rms = [], []
#     for tr, va in gkf.split(X, y, groups):
#         pipe.fit(X.iloc[tr], y[tr]); p = pipe.predict(X.iloc[va])
#         r2s.append(r2_score(y[va], p)); rms.append(np.sqrt(mean_squared_error(y[va], p)))
#     return np.mean(r2s), np.mean(rms)

# def run_cat(cols, cat_cols):
#     X = data[cols].copy()
#     cats = [c for c in cat_cols if c in cols]
#     cat_idx = [X.columns.get_loc(c) for c in cats]
#     r2s, rms = [], []
#     for tr, va in gkf.split(X, y, groups):
#         m = CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_state=42, verbose=0)
#         m.fit(Pool(X.iloc[tr], y[tr], cat_features=cat_idx)); p = m.predict(X.iloc[va])
#         r2s.append(r2_score(y[va], p)); rms.append(np.sqrt(mean_squared_error(y[va], p)))
#     return np.mean(r2s), np.mean(rms)

# featuresets = [
#     ("SAR only",             sar_cols,                              []),
#     ("SAR + physics",        sar_cols + physics_cols,               []),
#     ("physics only",         physics_cols,                          []),
#     ("attr + SAR + physics", attr_features + sar_cols + physics_cols, attr_cat),
# ]

# print(f"{'feature set':<24}{'model':<20}{'R2':>8}{'RMSE':>8}")
# print("="*60)
# for fs_name, cols, cats in featuresets:
#     for name, mdl in [("Linear Regression", LinearRegression()),
#                       ("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))]:
#         r2, rm = run_sklearn(mdl, cols, cats)
#         print(f"{fs_name:<24}{name:<20}{r2:>8.3f}{rm:>8.2f}")
#     r2, rm = run_cat(cols, cats)
#     print(f"{fs_name:<24}{'CatBoost':<20}{r2:>8.3f}{rm:>8.2f}")
#     print("-"*60)

In [6]:
# import numpy as np
# from sklearn.model_selection import GroupKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import NearestNeighbors
# from sklearn.metrics import r2_score
# from catboost import CatBoostRegressor, Pool

# K = 15
# attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
# attr_num = ["area_acres", "year"]
# model_cols = attr_cat + attr_num + sar_cols
# sim_cols = sar_cols + ["area_acres"]   # what makes two fields "similar"

# data = d.reset_index(drop=True).copy()
# for c in attr_cat:
#     data[c] = data[c].astype(str).fillna("Unknown")

# X = data[model_cols]
# y = data["crop_yield"].values
# years = data["year"].values
# groups = data["geo_group"].values
# cat_idx = [X.columns.get_loc(c) for c in attr_cat]

# def knn_denoise(tr, va):
#     # neighbours come from TRAIN rows only, and only from the same year
#     scaler = StandardScaler().fit(data.loc[tr, sim_cols])
#     Z = scaler.transform(data[sim_cols])
#     y_tr_dn = np.full(len(tr), np.nan)
#     y_va_dn = np.full(len(va), np.nan)
#     for yr in np.unique(years):
#         pool = tr[years[tr] == yr]
#         if len(pool) == 0:
#             continue
#         nn = NearestNeighbors(n_neighbors=min(K, len(pool))).fit(Z[pool])
#         for rows, out in [(tr, y_tr_dn), (va, y_va_dn)]:
#             pos = np.where(years[rows] == yr)[0]
#             if len(pos):
#                 _, idx = nn.kneighbors(Z[rows[pos]])
#                 out[pos] = y[pool][idx].mean(axis=1)
#     assert not np.isnan(y_tr_dn).any() and not np.isnan(y_va_dn).any()
#     return y_tr_dn, y_va_dn

# gkf = GroupKFold(n_splits=5)
# res = {"1. no denoising (real yield)": [],
#        "2. denoised: R2 vs denoised": [],
#        "3. denoised: R2 vs REAL yield": [],
#        "4. mean baseline vs denoised": []}

# for tr, va in gkf.split(X, y, groups):
#     # 1. our previous setup, for side by side
#     m = CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_state=42, verbose=0)
#     m.fit(Pool(X.iloc[tr], y[tr], cat_features=cat_idx))
#     res["1. no denoising (real yield)"].append(r2_score(y[va], m.predict(X.iloc[va])))

#     # 2-4. train on denoised yield
#     y_tr_dn, y_va_dn = knn_denoise(tr, va)
#     m = CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_state=42, verbose=0)
#     m.fit(Pool(X.iloc[tr], y_tr_dn, cat_features=cat_idx))
#     p = m.predict(X.iloc[va])
#     res["2. denoised: R2 vs denoised"].append(r2_score(y_va_dn, p))
#     res["3. denoised: R2 vs REAL yield"].append(r2_score(y[va], p))
#     res["4. mean baseline vs denoised"].append(r2_score(y_va_dn, np.full(len(va), y_tr_dn.mean())))

# print(f"kNN denoising, k={K}, same-year neighbours, train rows only, spatial 5-fold, n={len(data)}")
# print("-" * 62)
# for name, vals in res.items():
#     print(f"{name:<34} R2 {np.mean(vals):6.3f}  (+/- {np.std(vals):.3f})")

In [7]:
# import numpy as np
# from sklearn.model_selection import GroupKFold
# from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
# from catboost import CatBoostRegressor, Pool

# attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
# attr_num = ["area_acres", "year"]
# model_cols = attr_cat + attr_num + sar_cols

# data = d.reset_index(drop=True).copy()
# for c in attr_cat:
#     data[c] = data[c].astype(str).fillna("Unknown")

# X = data[model_cols]
# y = data["crop_yield"].values
# groups = data["geo_group"].values
# cat_idx = [X.columns.get_loc(c) for c in attr_cat]
# gkf = GroupKFold(n_splits=5)

# losses = {
#     "RMSE (normal)":   "RMSE",
#     "Huber delta=1.5": "Huber:delta=1.5",
#     "Huber delta=3":   "Huber:delta=3.0",
#     "MAE":             "MAE",
# }

# print("CatBoost, attributes + SAR, spatial 5-fold, scored on REAL yield, n =", len(data))
# print(f"{'loss':<18}{'R2':>8}{'RMSE':>8}{'MAE':>8}")
# print("-" * 42)
# for name, loss in losses.items():
#     r2s, rmses, maes = [], [], []
#     for tr, va in gkf.split(X, y, groups):
#         m = CatBoostRegressor(loss_function=loss, iterations=400, depth=6,
#                               learning_rate=0.05, random_state=42, verbose=0)
#         m.fit(Pool(X.iloc[tr], y[tr], cat_features=cat_idx))
#         p = m.predict(X.iloc[va])
#         r2s.append(r2_score(y[va], p))
#         rmses.append(np.sqrt(mean_squared_error(y[va], p)))
#         maes.append(mean_absolute_error(y[va], p))
#     print(f"{name:<18}{np.mean(r2s):>8.3f}{np.mean(rmses):>8.2f}{np.mean(maes):>8.2f}")

In [8]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
attr_num = ["area_acres", "year"]
num_cols = attr_num + sar_cols

data = d.reset_index(drop=True).copy()
for c in attr_cat:
    data[c] = data[c].astype(str).fillna("Unknown")

X = data[attr_cat + num_cols]
y = data["crop_yield"].values
groups = data["geo_group"].values
gkf = GroupKFold(n_splits=5)
K_LIST = ["all", 30, 15]

# model used to RANK features inside RFE (normal loss, fast settings)
def selector(family):
    if family == "Linear":
        return LinearRegression()
    if family == "RF":
        return RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    return CatBoostRegressor(iterations=200, depth=6, learning_rate=0.05, random_state=42, verbose=0)

# final model, with normal loss or MAE loss
def final_model(family, loss):
    if family == "Linear":
        return LinearRegression() if loss == "standard" else QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")
    if family == "RF":
        return (RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1) if loss == "standard"
                else RandomForestRegressor(n_estimators=200, criterion="absolute_error", random_state=42, n_jobs=-1))
    return CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_state=42, verbose=0,
                             loss_function="RMSE" if loss == "standard" else "MAE")

results = []
t0 = time.time()
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
    # encoding and scaling fitted on the training fold only
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), attr_cat),
        ("num", StandardScaler(), num_cols),
    ])
    Z_tr = prep.fit_transform(X.iloc[tr])
    Z_va = prep.transform(X.iloc[va])
    n_cols = Z_tr.shape[1]

    for family in ["Linear", "RF", "CatBoost"]:
        # rank all columns once per fold, on the training fold only
        rfe = RFE(selector(family), n_features_to_select=1, step=0.1).fit(Z_tr, y[tr])
        order = np.argsort(rfe.ranking_, kind="stable")   # best-ranked columns first
        for k in K_LIST:
            cols = np.arange(n_cols) if k == "all" else order[:k]
            for loss in ["standard", "MAE"]:
                m = final_model(family, loss).fit(Z_tr[:, cols], y[tr])
                p = m.predict(Z_va[:, cols])
                results.append({"model": family, "features": str(k), "loss": loss, "fold": fold,
                                "R2": r2_score(y[va], p),
                                "RMSE": np.sqrt(mean_squared_error(y[va], p)),
                                "MAE": mean_absolute_error(y[va], p)})
    print(f"fold {fold} done ({time.time() - t0:.0f} s)")

res = pd.DataFrame(results)
summary = (res.groupby(["model", "features", "loss"])
              .agg(R2=("R2", "mean"), R2_sd=("R2", "std"), RMSE=("RMSE", "mean"), MAE=("MAE", "mean"))
              .round(3).reset_index())
summary["f_order"] = summary["features"].map({"all": 0, "30": 1, "15": 2})
summary = summary.sort_values(["model", "f_order", "loss"]).drop(columns="f_order")

print(f"\n{n_cols} encoded columns | n = {len(data)} fields | spatial 5-fold CV")
print(summary.to_string(index=False))

fold 1 done (57 s)
fold 2 done (112 s)
fold 3 done (160 s)
fold 4 done (254 s)
fold 5 done (344 s)

67 encoded columns | n = 1197 fields | spatial 5-fold CV
   model features     loss     R2  R2_sd  RMSE   MAE
CatBoost      all      MAE  0.094  0.041 2.882 1.835
CatBoost      all standard  0.089  0.061 2.873 1.877
CatBoost       30      MAE  0.081  0.041 2.902 1.849
CatBoost       30 standard  0.091  0.050 2.872 1.870
CatBoost       15      MAE  0.086  0.028 2.887 1.868
CatBoost       15 standard  0.069  0.037 2.906 1.907
  Linear      all      MAE  0.071  0.049 2.919 1.864
  Linear      all standard  0.047  0.074 2.929 1.940
  Linear       30      MAE  0.056  0.047 2.936 1.897
  Linear       30 standard  0.045  0.065 2.936 1.947
  Linear       15      MAE  0.062  0.044 2.921 1.893
  Linear       15 standard  0.044  0.066 2.936 1.942
      RF      all      MAE  0.062  0.048 2.917 1.896
      RF      all standard -0.030  0.092 3.043 1.953
      RF       30      MAE  0.062  0.053 2.915 1

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor, Pool

ROOT = Path(r"F:\THESIS\mtech_thesis_crop_yield_estimation")

attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
attr_num = ["area_acres", "year"]
num_cols = attr_num + sar_cols

data = d.reset_index(drop=True).copy()
for c in attr_cat:
    data[c] = data[c].astype(str).fillna("Unknown")

X = data[attr_cat + num_cols]
y = data["crop_yield"].values
groups = data["geo_group"].values
gkf = GroupKFold(n_splits=5)
cat_idx = [X.columns.get_loc(c) for c in attr_cat]

# training losses to compare (same CatBoost settings for all)
losses = {
    "Squared error (normal)": "RMSE",
    "Huber, delta = 1.5":     "Huber:delta=1.5",
    "Huber, delta = 3":       "Huber:delta=3.0",
    "Absolute error":         "MAE",
}

def make_model(loss):
    return CatBoostRegressor(loss_function=loss, iterations=400, depth=6,
                             learning_rate=0.05, random_state=42, verbose=0)

rows = []
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
    X_tr, X_va = X.iloc[tr], X.iloc[va]

    # one-hot: encoder fitted on the training fold only
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(X_tr[attr_cat])
    Z_tr = np.hstack([ohe.transform(X_tr[attr_cat]), X_tr[num_cols].values])
    Z_va = np.hstack([ohe.transform(X_va[attr_cat]), X_va[num_cols].values])

    for loss_name, loss in losses.items():
        for enc in ["Native", "One-hot"]:
            m = make_model(loss)
            if enc == "Native":   # CatBoost reads the text categories directly
                m.fit(Pool(X_tr, y[tr], cat_features=cat_idx))
                p = m.predict(X_va)
            else:                 # categories as 0/1 columns
                m.fit(Z_tr, y[tr])
                p = m.predict(Z_va)
            rows.append({"encoding": enc, "loss": loss_name, "fold": fold,
                         "R2": r2_score(y[va], p),
                         "RMSE": np.sqrt(mean_squared_error(y[va], p)),
                         "mean_abs_error": mean_absolute_error(y[va], p)})
    print(f"fold {fold} done")

res = pd.DataFrame(rows)
summary = (res.groupby(["encoding", "loss"])
              .agg(R2=("R2", "mean"), R2_sd=("R2", "std"),
                   RMSE=("RMSE", "mean"), mean_abs_error=("mean_abs_error", "mean"))
              .round(3).reset_index())
summary["loss"] = pd.Categorical(summary["loss"], list(losses.keys()), ordered=True)
summary = summary.sort_values(["encoding", "loss"])

print(f"\nCatBoost, attributes + SAR, n = {len(data)} fields, spatial 5-fold CV")
print(summary.to_string(index=False))
print("\nR2 side by side:")
print(summary.pivot(index="loss", columns="encoding", values="R2").to_string())

# save for the slides
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / "catboost_encoding_x_loss_folds.csv", index=False)
summary.to_csv(out_dir / "catboost_encoding_x_loss_summary.csv", index=False)
print("\nsaved to:", out_dir)

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done

CatBoost, attributes + SAR, n = 1197 fields, spatial 5-fold CV
encoding                   loss     R2  R2_sd  RMSE  mean_abs_error
  Native Squared error (normal)  0.072  0.069 2.900           1.904
  Native     Huber, delta = 1.5 -0.020  0.070 3.041           2.042
  Native       Huber, delta = 3 -0.011  0.079 3.017           2.030
  Native         Absolute error  0.075  0.030 2.910           1.859
 One-hot Squared error (normal)  0.089  0.061 2.873           1.877
 One-hot     Huber, delta = 1.5  0.050  0.050 2.932           1.926
 One-hot       Huber, delta = 3  0.037  0.068 2.950           1.966
 One-hot         Absolute error  0.093  0.040 2.883           1.834

R2 side by side:
encoding                Native  One-hot
loss                                   
Squared error (normal)   0.072    0.089
Huber, delta = 1.5      -0.020    0.050
Huber, delta = 3        -0.011    0.037
Absolute error           0.075    0.093

saved

In [10]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

ROOT = Path(r"F:\THESIS\mtech_thesis_crop_yield_estimation")

attr_cat = ["irrigation", "taluka", "variety", "soil_texture", "soil_depth", "soil_color", "soil_structure"]
attr_num = ["area_acres", "year"]
num_cols = attr_num + sar_cols

data = d.reset_index(drop=True).copy()
for c in attr_cat:
    data[c] = data[c].astype(str).fillna("Unknown")

X = data[attr_cat + num_cols]
y = data["crop_yield"].values
groups = data["geo_group"].values
gkf = GroupKFold(n_splits=5)

def make_models():
    xgb = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
               colsample_bytree=0.8, random_state=42, n_jobs=-1)
    return {
        ("XGBoost", "Squared error"):  XGBRegressor(objective="reg:squarederror", **xgb),
        ("XGBoost", "Absolute error"): XGBRegressor(objective="reg:absoluteerror", **xgb),
        ("MLP (64-32-16)", "Squared error"): TransformedTargetRegressor(
            regressor=MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation="relu", alpha=1e-3,
                                   learning_rate_init=1e-3, max_iter=500, early_stopping=True,
                                   validation_fraction=0.15, n_iter_no_change=20, random_state=42),
            transformer=StandardScaler()),
    }

rows = []
t0 = time.time()
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
    # encoding and scaling fitted on the training fold only
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), attr_cat),
        ("num", StandardScaler(), num_cols),
    ])
    Z_tr = prep.fit_transform(X.iloc[tr])
    Z_va = prep.transform(X.iloc[va])

    for (name, loss), model in make_models().items():
        model.fit(Z_tr, y[tr])
        p = model.predict(Z_va)
        rows.append({"model": name, "loss": loss, "fold": fold,
                     "R2": r2_score(y[va], p),
                     "RMSE": np.sqrt(mean_squared_error(y[va], p)),
                     "mean_abs_error": mean_absolute_error(y[va], p)})
    print(f"fold {fold} done ({time.time() - t0:.0f} s)")

res = pd.DataFrame(rows)
summary = (res.groupby(["model", "loss"])
              .agg(R2=("R2", "mean"), R2_sd=("R2", "std"),
                   RMSE=("RMSE", "mean"), mean_abs_error=("mean_abs_error", "mean"))
              .round(3).reset_index())

print(f"\nAttributes + SAR, one-hot, n = {len(data)} fields, spatial 5-fold CV")
print(summary.to_string(index=False))
print("\nFor reference (previous run): CatBoost one-hot, absolute error: R2 0.093 \u00B1 0.040, mean abs. error 1.834")

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / "xgboost_mlp_folds.csv", index=False)
summary.to_csv(out_dir / "xgboost_mlp_summary.csv", index=False)
print("saved to:", out_dir)

fold 1 done (2 s)
fold 2 done (8 s)
fold 3 done (14 s)
fold 4 done (21 s)
fold 5 done (27 s)

Attributes + SAR, one-hot, n = 1197 fields, spatial 5-fold CV
         model           loss    R2  R2_sd  RMSE  mean_abs_error
MLP (64-32-16)  Squared error 0.063  0.048 2.920           1.950
       XGBoost Absolute error 0.090  0.038 2.885           1.843
       XGBoost  Squared error 0.025  0.081 2.964           1.946

For reference (previous run): CatBoost one-hot, absolute error: R2 0.093 ± 0.040, mean abs. error 1.834
saved to: F:\THESIS\mtech_thesis_crop_yield_estimation\results
